# CLEANING WITHDRAWAL VARIABLES

Data Cleaning is the first step of any Data Science project and can sometimes be very time consuming.
It involves identifying and correcting errors, inconsistencies or incomplete data within a dataset.

1. Handle missing values
2. Remove duplicates/irrelevant or hard to preprocess variables
3. Clean text data with an incorrect format

In [ ]:
pip install matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from utils import *

In [ ]:
df_train = pd.read_csv('../X_train_Hi5.csv')
df_test = pd.read_csv('../X_test_Hi5.csv')


In [ ]:
df = df_train.filter(regex='^prelev')
df['piezo_groundwater_level_category'] = df_train['piezo_groundwater_level_category']

In [ ]:
# Affichage des noms des variables
variables = df.columns.tolist()

The `.info()` function allows you to check:

1. The number of "non-null" values (number of rows without a NaN value)
2. The data type (or dtype of each column)
3. The number of rows and the columns present in the dataset

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

## 1. Data Processing

Real-word data often contains missing values. This can be due to technical failures, data entry issues, incompatible data formats or even intentional omissions (for privacy concerns). Examples include a lack of response to a survey or equipment failures when collecting data.

Here is a list of strategies to handle missing values:

- Replace NaN with the mean or median value (for continuous data)
- Replace NaN with the most frequent value (for categorical data)
- Create a new category for the missing values (for categorical data)
- Use a Machine Learning model to predict the missing values (KNN, Random Forest,...)

### Selection of variables with missing values

This code identifies and prints the names of columns that contain missing (NaN) values in the DataFrame, helping to detect and address missing data.

In [ ]:
missing_columns = df.columns[df.isna().sum() > 0].tolist()
print(missing_columns)

### Standardization of object-type columns

Standardization of `object-type` columns and columns names (not required here as long as features are already standardized)

In [ ]:
# Standardize object-type columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.lower().str.strip()

# Standardize column names to simplify their manipulation
df.rename(columns=lambda x: x.strip().lower().replace(' ', '_'), inplace=True)

### Duplicates

This removes duplicate rows from the DataFrame, ensuring that each row is unique. It's useful to avoid redundancy and improve the quality of the data before further analysis or modeling.

In [ ]:
df.drop_duplicates(inplace=True)

### Features analysis

In [ ]:
for var in variables:
    if df[var].dtype == 'object':
        print(len(df[var].unique()), var)

prelev_structure_code_0, prelev_structure_code_1 and prelev_structure_code_2 all have too much categories.

In [ ]:
# We take off prelev_structure_code_0, prelev_structure_code_1 and prelev_structure_code_2
keeped_features = ['prelev_volume_0', 'prelev_usage_label_0', 'prelev_volume_obtention_mode_label_0', 'prelev_longitude_0', 'prelev_latitude_0', 'prelev_commune_code_insee_0', 'prelev_volume_1', 'prelev_usage_label_1', 'prelev_volume_obtention_mode_label_1', 'prelev_longitude_1', 'prelev_latitude_1', 'prelev_commune_code_insee_1', 'prelev_volume_2', 'prelev_usage_label_2', 'prelev_volume_obtention_mode_label_2', 'prelev_longitude_2', 'prelev_latitude_2', 'prelev_commune_code_insee_2', 'prelev_other_volume_sum', 'piezo_groundwater_level_category']

### Categorical features processing

In [ ]:
df_clean = df[keeped_features]

# Encode categorical variables using LabelEncoder
le = LabelEncoder()
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = le.fit_transform(df_clean[col])

# Interesting Features Selection

This code aims to analyze the importance of certain features to predict `piezo_groundwater_level_category`.

In [ ]:
var = "piezo_groundwater_level_category"

# 1. Separate explanatory variables (X) and the target variable (y)
X = df_clean.drop(columns=[var])
y = df_clean[var]

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Apply Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Display feature importance
importances = model.feature_importances_

# Create a DataFrame to display the feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})

# Sort by descending importance
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

# Display the results
print(feature_importance)

# 5. Visualize feature importance as a bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance)
plt.title('Feature Importance with Random Forest')
plt.show()

# 6. Predict the target variable on the test set
y_pred = model.predict(X_test)

# 7. Calculate the F1 score
f1 = f1_score(y_test, y_pred, average='weighted')  # For multi-class classification, weighted average
print(f'F1 Score: {f1}')

We select the only relevant feature `prelev_other_volume_sum`. This allows us to focus on the key data while reducing noise and improving the model's performance.

In [ ]:
withdrawal_features = ['prelev_other_volume_sum']